## Set-up the project environment for Colibri Data Lakehouse
1. Create external location - dea_ext_dl_Colibri
1. Create Catalog - Colibri
1. Create Schemas
    - landing
    - Bronze
    - Silver
    - Gold
1. Create Volume - operational_data

### 1. Create External Location
**External Location Name:** dea_course_ext_dl_circuitbox  
_ADLS Path:_ [abfss://circuitbox@deacourseextdl.dfs.core.windows.net/](abfss://circuitbox@deacourseextdl.dfs.core.windows.net/)  
_Storage Credential:_ dea_course_ext_sc

In [0]:
%sql
DROP CATALOG IF EXISTS colibri CASCADE

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS colibri_test
 MANAGED LOCATION 's3://osasa-s3-demo-bucket/'

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8025005925827097>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "CREATE CATALOG IF NOT EXISTS colibri_test\n MANAGED LOCATION 's3://osasa-s3-demo-bucket/'\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:192, in SqlMagic.sql(self, line, cell)
    186 except BaseExce

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS colibrix

In [0]:
%sql
USE CATALOG colibri_test;
CREATE SCHEMA IF NOT EXISTS landing

In [0]:
%sql
USE CATALOG colibri_test;
CREATE SCHEMA IF NOT EXISTS silver

In [0]:
%fs
ls 's3://osasa-s3-demo-bucket/operational_data/'

In [0]:
df = spark.read.format("csv").option("header","true").option("inferSchema","true").load('/Volumes/colibri_test/default/test/operational_data/colibric/')


In [0]:
%python
df= spark.read.format("csv").option("header","true").option("inferSchema","true").load('s3://osasa-s3-demo-bucket/operational_data/colibric/')
display(df)

In [0]:
%python
schema = """timestamp timestamp,turbine_id int,wind_speed double,wind_direction int,power_output double"""

df = (spark.read.option("header", "true").schema(schema).csv('s3://osasa-s3-demo-bucket/operational_data/colibric/'))


In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window
w = Window.partitionBy("turbine_id")

clean_1  = (df.withColumn("timestamp", F.to_timestamp("timestamp")).withColumn("power_output", F.col("power_output").cast("double")).dropDuplicates(["turbine_id", "timestamp"]).withColumn("dq_missing_power", F.col("power_output").isNull()))



clean = (clean_1.withColumn("power_median",F.expr("percentile_approx(power_output, 0.5)").over(w)) \
    .withColumn("power_output_clean",F.coalesce(F.col("power_output"), F.col("power_median"))) \
        .withColumn("date", F.to_date("timestamp")))




In [0]:
daily_stats = (clean.groupBy("date", "turbine_id").agg(F.min("power_output_clean").alias("min_power_mw"),F.max("power_output_clean").alias("max_power_mw"),F.avg("power_output_clean").alias("avg_power_mw"),F.stddev("power_output_clean").alias("std_power_mw"),F.count("*").alias("reading_count")))
display(daily_stats)

In [0]:
from pyspark.sql.functions import count, min, max, avg, stddev

daily_stats = (clean.groupBy("date", "turbine_id").agg(
    min("power_output_clean").alias("min_power_mw"),
    max("power_output_clean").alias("max_power_mw"),
    avg("power_output_clean").alias("avg_power_mw"),
    stddev("power_output_clean").alias("std_power_mw"),
    count("*").alias("reading_count")
))
display(daily_stats)

In [0]:
from pyspark.sql.functions import count, min, max, avg, stddev
daily_stats = (transformed.groupBy("date", "turbine_id").agg(
min("power_output_clean").alias("min_power_mw"),
max("power_output_clean").alias("max_power_mw"),
avg("power_output_clean").alias("avg_power_mw"),
stddev("power_output_clean").alias("std_power_mw"),
count("*").alias("reading_count")
))
display(daily_stats)


In [0]:
fleet = daily_stats.groupBy("date").agg(F.avg("avg_power_mw").alias("fleet_mean"),F.stddev("avg_power_mw").alias("fleet_std"))
result = (daily_stats.join(fleet, "date").withColumn("lower", F.col("fleet_mean") - 2*F.col("fleet_std")).withColumn("upper", F.col("fleet_mean") + 2*F.col("fleet_std")).withColumn("is_anomaly",(F.col("avg_power_mw") < F.col("lower")) |(F.col("avg_power_mw") > F.col("upper"))))



In [0]:
daily_stats.write.format("delta") \
    .mode("append") \
        .partitionBy("date") \
            .saveAsTable("gizmobox.silver.turbine_daily_stats")

In [0]:
%sql
select * from gizmobox.silver.turbine_daily_stats

In [0]:
fleet = daily_stats.groupBy("date").agg(F.avg("avg_power_mw").alias("fleet_mean"),F.stddev("avg_power_mw").alias("fleet_std"))

result = (daily_stats.join(fleet, "date").withColumn("lower", F.col("fleet_mean") - 2*F.col("fleet_std")).withColumn("upper", F.col("fleet_mean") + 2*F.col("fleet_std")).withColumn("is_anomaly",(F.col("avg_power_mw") < F.col("lower")) |(F.col("avg_power_mw") > F.col("upper"))))


In [0]:
result.write.format("delta") \
    .mode("append") \
        .partitionBy("date") \
            .saveAsTable("gizmobox.silver.turbine_daily_fleet_result")

In [0]:
%sql
select * from gizmobox.silver.turbine_daily_fleet_result

date,turbine_id,min_power_mw,max_power_mw,avg_power_mw,std_power_mw,reading_count,fleet_mean,fleet_std,lower,upper,is_anomaly
2022-03-18,1,1.6,4.5,3.1041666666666674,0.9119873982094016,24,3.0047222222222225,0.1342219889885495,2.7362782442451237,3.2731662001993214,false
2022-03-18,2,1.5,4.3,2.7375000000000003,0.8313490869351009,24,3.0047222222222225,0.1342219889885495,2.7362782442451237,3.2731662001993214,false
2022-03-18,3,1.5,4.5,2.8958333333333326,0.9308617637374551,24,3.0047222222222225,0.1342219889885495,2.7362782442451237,3.2731662001993214,false
2022-03-18,4,1.6,4.5,2.954166666666666,0.9753390343280937,24,3.0047222222222225,0.1342219889885495,2.7362782442451237,3.2731662001993214,false
2022-03-18,5,1.5,4.2,2.9083333333333337,0.8672378274010897,24,3.0047222222222225,0.1342219889885495,2.7362782442451237,3.2731662001993214,false
2022-03-18,6,1.5,4.3,3.095833333333333,0.8725171814262789,24,3.0047222222222225,0.1342219889885495,2.7362782442451237,3.2731662001993214,false
2022-03-18,7,1.6,4.4,3.0208333333333335,0.856676676194836,24,3.0047222222222225,0.1342219889885495,2.7362782442451237,3.2731662001993214,false
2022-03-18,8,1.5,4.4,2.9166666666666674,1.0327955589886446,24,3.0047222222222225,0.1342219889885495,2.7362782442451237,3.2731662001993214,false
2022-03-18,9,1.6,4.5,2.929166666666666,0.8477485100145468,24,3.0047222222222225,0.1342219889885495,2.7362782442451237,3.2731662001993214,false
2022-03-18,10,1.9,4.3,3.141666666666667,0.7734264061933117,24,3.0047222222222225,0.1342219889885495,2.7362782442451237,3.2731662001993214,false


In [0]:
src/
 ingestion.py        # discover + read CSVs  
 quality.py          # schema, ranges, missing data, dedupe  
 transform.py        # daily summaries 
 anomaly.py          # 2σ rule  
 storage.py          # Delta/database writes 
tests/
 test_quality.py  
 test_transform.py
 test_anomaly.py
jobs/
 daily_pipeline.py



In [0]:
# Example assertions 
assert transformed.filter(F.col("turbine_id").isNull()).count() == 0
assert daily_stats.filter(F.col("reading_count") > 24).count() == 0
# Inject a deliberately extreme turbine-day
# # and assert is_anomaly == True.

